# Notebook 2: Feature Engineering
Build the full feature matrix that captures degradation signals for the ML models.

## 1. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

from src.ingestion.data_loader import load_train_test, INFORMATIVE_SENSORS
from src.features.signal_processing import (
    compute_rms, compute_crest_factor, compute_kurtosis,
    compute_rolling_features, add_cross_sensor_features,
    normalize_per_unit, WINDOW_SIZES
)
from src.features.label_engineering import (
    add_rul_labels, add_binary_labels, add_lifecycle_features, print_label_stats
)
print("Ready.")

## 2. Load and Inspect Raw Data

In [ ]:
train_df, test_df, true_rul = load_train_test('FD001')
print(f"Train shape: {train_df.shape}")
print(f"Units: {train_df['unit_id'].nunique()}")
train_df[INFORMATIVE_SENSORS].describe().round(2)

## 3. RUL Label Engineering
Work backwards from each failure event to assign a Remaining Useful Life label.

In [ ]:
train_labeled = add_rul_labels(train_df.copy(), max_rul=125)
train_labeled = add_binary_labels(train_labeled)
train_labeled = add_lifecycle_features(train_labeled)

print_label_stats(train_labeled)

# Visualise RUL distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(train_labeled['rul'], bins=40, color='#378ADD', edgecolor='white')
axes[0].set_title('RUL Distribution (uncapped)')
axes[0].set_xlabel('Remaining Useful Life (cycles)')

axes[1].hist(train_labeled['rul_capped'], bins=40, color='#1D9E75', edgecolor='white')
axes[1].axvline(125, color='#E24B4A', linestyle='--', label='Cap = 125')
axes[1].set_title('RUL Distribution (capped at 125)')
axes[1].set_xlabel('RUL capped')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Per-Unit Normalisation
Remove unit-to-unit manufacturing variance — we only want to model degradation, not baseline differences.

In [ ]:
train_norm = normalize_per_unit(train_labeled.copy(), INFORMATIVE_SENSORS[:4])

sensor = 'sensor_2'
unit_sample = [1, 2, 3]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for uid in unit_sample:
    u = train_norm[train_norm['unit_id'] == uid].sort_values('cycle')
    axes[0].plot(u['cycle'], u[sensor], alpha=0.7, label=f'Unit {uid}')
    axes[1].plot(u['cycle'], u[f'{sensor}_normalized'], alpha=0.7, label=f'Unit {uid}')

axes[0].set_title(f'{sensor} — raw values')
axes[1].set_title(f'{sensor} — per-unit normalized')
for ax in axes:
    ax.set_xlabel('Cycle')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("Normalized sensor has zero mean at baseline — degradation trend preserved.")

## 5. Rolling Statistical Features
Capture time-domain degradation signals: RMS, kurtosis, crest factor.

In [ ]:
from scipy.stats import kurtosis

# Demo: compute rolling kurtosis for one unit
unit1 = train_norm[train_norm['unit_id'] == 1].sort_values('cycle').copy()
unit1_labeled = add_rul_labels(unit1)

vals = unit1['sensor_20_normalized'].fillna(0).values
kurtosis_30 = pd.Series(vals).rolling(30, min_periods=1).apply(
    lambda x: kurtosis(x, fisher=False) if len(x) >= 4 else 3.0, raw=True
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(unit1['cycle'], unit1['sensor_20_normalized'], color='#378ADD', linewidth=1.2)
axes[0].set_ylabel('Normalized value')
axes[0].set_title('sensor_20 (vibration radial) — normalized')

axes[1].plot(unit1['cycle'], kurtosis_30, color='#EF9F27', linewidth=1.5)
axes[1].axhline(6, color='#E24B4A', linestyle='--', label='Fault threshold (kurtosis=6)')
axes[1].set_xlabel('Operating cycle')
axes[1].set_ylabel('Kurtosis (30-cycle window)')
axes[1].set_title('Rolling kurtosis — rises as bearing degrades')
axes[1].legend()

plt.tight_layout()
plt.show()
print("Kurtosis rises sharply in final 20–30 cycles — early warning signal.")

## 6. Cross-Sensor Features

In [ ]:
train_cross = add_cross_sensor_features(train_norm)

new_features = ['feat_temp_press_ratio', 'feat_temp_diff',
                'feat_flow_efficiency', 'feat_vibration_total']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
train_cross_labeled = add_rul_labels(train_cross)

for ax, feat in zip(axes.flatten(), new_features):
    sample = train_cross_labeled.sample(min(3000, len(train_cross_labeled)), random_state=42)
    sc = ax.scatter(sample['rul'], sample[feat], alpha=0.15, s=5, c=sample['rul'],
                    cmap='RdYlGn')
    ax.set_xlabel('RUL (cycles)')
    ax.set_ylabel(feat)
    ax.set_title(feat)

plt.suptitle('Cross-sensor features vs RUL', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7. Run Full Feature Pipeline

In [ ]:
# Run the complete feature pipeline from command line:
# python src/features/feature_pipeline.py

# Or run it here:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../src/features/feature_pipeline.py', '--subset', 'FD001'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[:500])

## 8. Feature Matrix Summary

In [ ]:
from src.features.feature_pipeline import load_features

data = load_features('FD001')
train_feat = data['train_df']
feature_cols = data['feature_cols']

print(f"Final feature matrix: {train_feat.shape}")
print(f"Number of features:   {len(feature_cols)}")
print(f"\nFeature categories:")
categories = {}
for c in feature_cols:
    cat = ('rolling' if any(f'_w{w}_' in c for w in [5,15,30])
           else 'normalized' if '_normalized' in c
           else 'cross_sensor' if c.startswith('feat_')
           else 'lifecycle' if 'cycle' in c or 'life' in c
           else 'other')
    categories[cat] = categories.get(cat, 0) + 1
for k, v in sorted(categories.items()):
    print(f"  {k:<20} {v}")

## Next step
Features are saved to `data/features/`. Proceed to model training: `03_model_training.ipynb`